## Data validation y data integrity

### By: Carlos Javier Palacios Sanchez

### Date: 06/09/2026

### Description:

Requerimiento
Modificar `feature_pipeline.py` para validar los datos antes de persistir los features.
El proceso debe garantizar calidad, consistencia, formato e integridad de los datos.

El proceso debe incluir:

- Reglas sobre tipos esperados, rangos, porcentaje máximo de nulos, categorías válidas,
  formatos de fecha y unicidad de campos clave.
- Reglas de integridad entre campos, registros y datasets cuando correspondan al problema.
- Implementación con la herramienta que quiera, ya sea Great Expectations, Pandera o
  validaciones manuales con pandas.
- Un error claro y ausencia de persistencia de features cuando falle una validación.

Entregables:

- `feature_pipeline.py` con validaciones documentadas.
- Pruebas unitarias con datos válidos e inválidos.

---

## Estrategia

### Por qué Pandera

De las tres opciones, **Pandera** es la que mejor encaja con este proyecto. Great
Expectations aporta un catálogo de expectativas y reportes HTML, pero arrastra una
configuración y unas dependencias desproporcionadas para un dataset de 480 filas.
Las validaciones manuales con pandas no añaden dependencias, pero dispersan las
reglas en decenas de `if` difíciles de leer y de mantener.

Pandera permite declarar el contrato de datos como un objeto —tipos, rangos,
categorías, nulos— que se lee de un vistazo y sirve a la vez de documentación. Lo
que Pandera no expresa con comodidad, que son las reglas que **cruzan** campos o
datasets, se implementa como funciones explícitas: así el mensaje de error puede
decir exactamente qué relación se rompió, en vez de un genérico "el check falló".

### Dónde se valida

El pipeline valida en tres puntos, y ninguno escribe nada:

| Punto | Función | Qué comprueba |
|---|---|---|
| Entrada | `validar_entrada` | esquema del archivo crudo, archivo no vacío |
| Intermedio | `validar_intermedio` | tipos, rangos, categorías, % de nulos, fechas, unicidad |
| Features | `validar_features` | tipos, infinitos, integridad entre campos y entre datasets |

La escritura ocurre **después** de las tres, en `ejecutar_pipeline`. Ese orden es la
garantía de que un dataset inválido no deja archivos a medias en `data/`.

## 1. Configuración

In [1]:
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd


def localizar_raiz() -> Path:
    """Sube por el árbol de directorios hasta encontrar el pyproject.toml."""
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pyproject.toml").is_file():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto")


RAIZ = localizar_raiz()
SCRIPT = RAIZ / "src" / "pipelines" / "feature_pipeline" / "feature_pipeline.py"
PRUEBAS = RAIZ / "tests" / "pipelines" / "feature_pipeline"
ENTRADA = RAIZ / "data" / "01_raw" / "corazon.csv"

sys.path.insert(0, str(RAIZ / "src"))

from pipelines.feature_pipeline import feature_pipeline as fp  # noqa: E402

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

print(f"Raíz del proyecto: {RAIZ}")
print(f"Pandera disponible: {fp.pa.__name__}")

Raíz del proyecto: /mnt/c/Users/KATANA/Heart_project
Pandera disponible: pandera.pandas


## 2. Catálogo de reglas

Las reglas no están dispersas por el código: se declaran como constantes al inicio
del script, de modo que ajustar un umbral no obliga a leer la implementación.

In [2]:
rangos = pd.DataFrame(
    [(col, minimo, maximo) for col, (minimo, maximo) in fp.RANGOS_VALIDOS.items()],
    columns=["columna", "mínimo", "máximo"],
)
print("Rangos de plausibilidad clínica:")
print(rangos.to_string(index=False))

print(f"\nProporción máxima de nulos por columna: {fp.MAX_PROPORCION_NULOS:.0%}")
print(f"Columnas de fecha declaradas: {fp.COLS_FECHA or 'ninguna (el dataset no tiene fechas)'}")
print(f"Clave de unicidad: {fp.CLAVE_UNICIDAD or 'el registro completo (no hay ID de paciente)'}")

Rangos de plausibilidad clínica:
 columna  mínimo  máximo
     age    18.0   120.0
 rest_bp    60.0   260.0
    chol    80.0   700.0
  max_hr    50.0   220.0
old_peak     0.0    10.0
      ca     0.0     3.0
     fbs     0.0     1.0

Proporción máxima de nulos por columna: 35%
Columnas de fecha declaradas: ninguna (el dataset no tiene fechas)
Clave de unicidad: el registro completo (no hay ID de paciente)


### Formatos de fecha y unicidad de campos clave

El requerimiento pide reglas de formato de fecha y de unicidad de campos clave. Este
dataset clínico **no tiene columnas de fecha ni identificador de paciente**, así que
en lugar de inventar campos que no existen:

- `validar_formato_fechas` queda implementada y parametrizada por `COLS_FECHA`.
  Con el diccionario vacío la regla informa que no aplica; basta añadir
  `{"fecha_examen": "%Y-%m-%d"}` para que empiece a validar. Más abajo se demuestra
  funcionando sobre datos de ejemplo.
- La unicidad se evalúa sobre el **registro completo**, que es el criterio que aplica
  cuando no hay clave natural: tras la deduplicación no puede quedar ninguna fila
  repetida. `validar_unicidad` también acepta una clave explícita.

In [3]:
print("Esquema del dataset intermedio declarado con Pandera:\n")
esquema = fp.esquema_intermedio()
for nombre, columna in esquema.columns.items():
    descripcion = columna.description or ""
    print(f"  {nombre:<12} nullable={columna.nullable!s:<5} {descripcion}")

Esquema del dataset intermedio declarado con Pandera:

  age          nullable=True  numérica en [18.0, 120.0]
  rest_bp      nullable=True  numérica en [60.0, 260.0]
  chol         nullable=True  numérica en [80.0, 700.0]
  max_hr       nullable=True  numérica en [50.0, 220.0]
  old_peak     nullable=True  numérica en [0.0, 10.0]
  ca           nullable=True  numérica en [0.0, 3.0]
  fbs          nullable=True  numérica en [0.0, 1.0]
  sex          nullable=True  categórica en ['female', 'male']
  chest_pain   nullable=True  categórica en ['asymptomatic', 'nonanginal', 'nontypical', 'typical']
  rest_ecg     nullable=True  categórica en ['left ventricular hypertrophy', 'normal', 'st-t wave abnormality']
  exang        nullable=True  categórica en ['0', '1']
  slope        nullable=True  categórica en ['1', '2', '3']
  thal         nullable=True  categórica en ['fixed', 'normal', 'reversable']
  disease      nullable=False variable objetivo binaria, sin faltantes


## 3. Ejecución con datos válidos

Con el dataset real todas las reglas pasan y el pipeline persiste los tres archivos.

In [4]:
ejecucion = subprocess.run(  # noqa: S603
    [sys.executable, str(SCRIPT)],
    capture_output=True,
    text=True,
    check=False,
)

print(ejecucion.stderr or ejecucion.stdout)
print(f"Código de salida: {ejecucion.returncode}")

16:24:48 | INFO     | === Feature pipeline: inicio ===
16:24:48 | INFO     | Datos crudos leídos desde /mnt/c/Users/KATANA/Heart_project/data/01_raw/corazon.csv -> 3030 filas x 14 columnas
16:24:50 | INFO     | [entrada] esquema 'datos_crudos': OK
16:24:50 | INFO     | Saneamiento de tipos: 33 valores inválidos convertidos a NaN
16:24:51 | INFO     | Duplicados eliminados: 2462 filas (3030 -> 568)
16:24:51 | INFO     | Filas sin etiqueta eliminadas: 88 -> dataset final 480 filas
16:24:51 | INFO     | Features construidos: 27 columnas a partir de 14 columnas originales
16:24:51 | INFO     | [intermedio] esquema 'dataset_intermedio': OK
16:24:51 | INFO     | [intermedio] proporción de nulos: OK (máximo 24.2%, umbral 35%)
16:24:51 | INFO     | [intermedio] formato de fechas: no aplica (el dataset no tiene fechas)
16:24:51 | INFO     | [intermedio] unicidad de registros: OK
16:24:51 | INFO     | [features] esquema 'features': OK
16:24:51 | INFO     | [features] integridad de atributos deri

## 4. Comportamiento con datos inválidos

Cada regla se prueba rompiendo deliberadamente un dato válido. El resultado esperado
en todos los casos es la misma excepción, `ErrorDeValidacion`, con un mensaje que
identifica la regla y la fila culpable.

In [5]:
crudo = fp.leer_datos_crudos(ENTRADA)
depurado, features = fp.construir_features(crudo)

print(f"Dataset intermedio: {depurado.shape}")
print(f"Tabla de features : {features.shape}")

Dataset intermedio: (480, 14)
Tabla de features : (480, 27)


In [6]:
import numpy as np


def probar(descripcion: str, validacion) -> dict:
    """Ejecuta una validación que debe fallar y captura su mensaje."""
    try:
        validacion()
    except fp.ErrorDeValidacion as error:
        mensaje = str(error).replace("\n", " | ")
        return {"regla": descripcion, "detectado": "sí", "mensaje": mensaje[:110]}
    return {"regla": descripcion, "detectado": "NO", "mensaje": "la regla no saltó"}


def romper(datos: pd.DataFrame, **cambios) -> pd.DataFrame:
    """Devuelve una copia del dataset con los valores indicados alterados."""
    copia = datos.copy()
    for columna, valor in cambios.items():
        copia.loc[0, columna] = valor
    return copia


casos = [
    probar(
        "rango: edad de 300 años",
        lambda: fp.validar_intermedio(romper(depurado, age=300.0)),
    ),
    probar(
        "categoría: valor fuera del catálogo",
        lambda: fp.validar_intermedio(romper(depurado, thal="categoria_inventada")),
    ),
    probar(
        "objetivo: valor no binario",
        lambda: fp.validar_intermedio(romper(depurado, disease=7)),
    ),
    probar(
        "nulos: columna vacía por encima del umbral",
        lambda: fp.validar_proporcion_nulos(depurado.assign(chol=np.nan), "demo"),
    ),
    probar(
        "unicidad: registro duplicado",
        lambda: fp.validar_unicidad(
            pd.concat([depurado, depurado.head(1)], ignore_index=True), "demo"
        ),
    ),
    probar(
        "entrada: archivo sin filas",
        lambda: fp.validar_entrada(crudo.head(0)),
    ),
]

pd.DataFrame(casos)

,regla,detectado,mensaje
0,rango: edad de 300 años,sí,[intermedio] el dataset no cumple el esquema 'dataset_intermedio': | schema_...
1,categoría: valor fuera del catálogo,sí,[intermedio] el dataset no cumple el esquema 'dataset_intermedio': | schema_...
2,objetivo: valor no binario,sí,[intermedio] el dataset no cumple el esquema 'dataset_intermedio': | schema_...
3,nulos: columna vacía por encima del umbral,sí,[demo] columnas por encima del 35% de nulos: chol (100.0%)
4,unicidad: registro duplicado,sí,[demo] hay 1 filas duplicadas según el registro completo
5,entrada: archivo sin filas,sí,[entrada] el dataset no cumple el esquema 'datos_crudos': | column fa...


### Integridad entre campos y entre datasets

Estas reglas no miran una columna aislada, sino la relación entre varias. Son las que
detectan un desalineado de filas o una fórmula que cambió sin propagarse.

In [7]:
def romper_derivado(datos: pd.DataFrame) -> pd.DataFrame:
    copia = datos.copy()
    copia.loc[0, "fc_maxima_teorica"] = copia.loc[0, "fc_maxima_teorica"] + 5
    return copia


def romper_onehot(datos: pd.DataFrame) -> pd.DataFrame:
    copia = datos.copy()
    copia.loc[0, [f"thal_{c}" for c in fp.CATEGORIAS_VALIDAS["thal"]]] = 1.0
    return copia


casos_integridad = [
    probar(
        "entre campos: derivado incoherente con su fórmula",
        lambda: fp.validar_integridad_derivados(romper_derivado(features), "demo"),
    ),
    probar(
        "entre campos: one-hot con dos categorías activas",
        lambda: fp.validar_integridad_onehot(romper_onehot(features), "demo"),
    ),
    probar(
        "entre campos: infinito por división mal controlada",
        lambda: fp.validar_features(romper(features, ratio_chol_edad=np.inf), depurado),
    ),
    probar(
        "entre datasets: se perdió una fila en la transformación",
        lambda: fp.validar_consistencia_datasets(depurado, features.iloc[:-1]),
    ),
]

pd.DataFrame(casos_integridad)

,regla,detectado,mensaje
0,entre campos: derivado incoherente con su fórmula,sí,[demo] 'fc_maxima_teorica' no coincide con 220 - age
1,entre campos: one-hot con dos categorías activas,sí,[demo] la codificación one-hot de 'thal' es inconsistente: 1 filas no suman ...
2,entre campos: infinito por división mal controlada,sí,[features] el dataset no cumple el esquema 'features': | schema_context ...
3,entre datasets: se perdió una fila en la transformación,sí,[features] el número de filas no coincide: intermedio 480 vs. features 479


### Formato de fecha

Se demuestra sobre datos de ejemplo, ya que el dataset no tiene columnas de fecha.

In [8]:
fechas_ok = pd.DataFrame({"fecha_examen": ["2026-01-15", "2026-02-28", None]})
fechas_mal = pd.DataFrame({"fecha_examen": ["2026-01-15", "15/01/2026"]})
FORMATO = {"fecha_examen": "%Y-%m-%d"}

fp.validar_formato_fechas(fechas_ok, "demo", FORMATO)
print("Fechas con formato correcto: aceptadas\n")

pd.DataFrame(
    [
        probar(
            "formato: fecha en otro formato",
            lambda: fp.validar_formato_fechas(fechas_mal, "demo", FORMATO),
        )
    ]
)

Fechas con formato correcto: aceptadas



,regla,detectado,mensaje
0,formato: fecha en otro formato,sí,[demo] la columna 'fecha_examen' tiene 1 valores que no respetan el formato ...


## 5. Requisito clave: ausencia de persistencia

La prueba decisiva del requerimiento. Se ejecuta el pipeline completo sobre un CSV con
una edad imposible —un valor perfectamente numérico, que por tanto sobrevive al
saneamiento y sólo puede detenerlo la validación de rango— en un directorio temporal
vacío, y se comprueba qué quedó escrito.

In [9]:
temporal = Path(tempfile.mkdtemp())
datos_malos = pd.read_csv(ENTRADA, dtype=str)
datos_malos.loc[0, "age"] = "300"
entrada_mala = temporal / "corazon_invalido.csv"
datos_malos.to_csv(entrada_mala, index=False)

fallo = subprocess.run(  # noqa: S603
    [
        sys.executable,
        str(SCRIPT),
        "--entrada",
        str(entrada_mala),
        "--intermedio",
        str(temporal / "intermedio.parquet"),
        "--salida",
        str(temporal / "features.parquet"),
        "--metadatos",
        str(temporal / "metadatos.json"),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(fallo.stderr)
print(f"Código de salida: {fallo.returncode}")
print(f"Archivos en el directorio de salida: {[p.name for p in temporal.iterdir()]}")
print("\nSólo está el CSV de entrada: no se persistió ningún feature.")

16:24:59 | INFO     | === Feature pipeline: inicio ===
16:24:59 | INFO     | Datos crudos leídos desde /tmp/tmp4ldp9nhb/corazon_invalido.csv -> 3030 filas x 14 columnas
16:24:59 | INFO     | [entrada] esquema 'datos_crudos': OK
16:24:59 | INFO     | Saneamiento de tipos: 33 valores inválidos convertidos a NaN
16:24:59 | INFO     | Duplicados eliminados: 2461 filas (3030 -> 569)
16:24:59 | INFO     | Filas sin etiqueta eliminadas: 88 -> dataset final 481 filas
16:24:59 | INFO     | Features construidos: 27 columnas a partir de 14 columnas originales
16:24:59 | ERROR    | VALIDACIÓN FALLIDA - no se persistió ningún archivo
[intermedio] el dataset no cumple el esquema 'dataset_intermedio':
schema_context column                 check  check_number  failure_case  index
        Column    age in_range(18.0, 120.0)             0         300.0      0

Código de salida: 1
Archivos en el directorio de salida: ['corazon_invalido.csv']

Sólo está el CSV de entrada: no se persistió ningún feature.


El mensaje es explícito y accionable: nombra la etapa (`intermedio`), el esquema, la
columna (`age`), la regla (`in_range(18.0, 120.0)`), el valor culpable (`300.0`) y su
índice de fila. No lleva traza de Python porque el fallo es de datos, no de código:
quien lo lee necesita corregir el archivo, no depurar el script.

## 6. Trazabilidad

El manifiesto que acompaña a los features documenta qué reglas superó el dataset que
sí se persistió.

In [10]:
import json as _json

manifiesto = _json.loads(
    (RAIZ / "data" / "04_feature" / "corazon_features_metadata.json").read_text(encoding="utf-8")
)

print(f"Generado en: {manifiesto['generado_en']}")
print(f"Dimensión  : {manifiesto['n_filas']} x {manifiesto['n_columnas']}\n")
print("Validaciones superadas:")
for regla in manifiesto["validaciones_superadas"]:
    print(f"  - {regla}")

Generado en: 2026-09-06T21:24:51+00:00
Dimensión  : 480 x 27

Validaciones superadas:
  - entrada: esquema de columnas y archivo no vacío
  - intermedio: tipos esperados por columna
  - intermedio: rangos de plausibilidad clínica
  - intermedio: categorías válidas del dominio
  - intermedio: variable objetivo binaria y sin faltantes
  - intermedio: proporción de nulos <= 35% por columna
  - intermedio: formato de las columnas de fecha declaradas
  - intermedio: unicidad de registros
  - features: todo numérico, finito y one-hot en {0, 1}
  - features: integridad entre campos (atributos derivados)
  - features: integridad entre campos (codificación one-hot)
  - features: integridad entre datasets (intermedio vs. features)


## 7. Pruebas unitarias

La batería cubre datos válidos y datos inválidos: para cada regla hay al menos un caso
que pasa y uno que falla, más dos pruebas que verifican que un fallo de validación no
deja ningún archivo escrito.

In [11]:
pruebas = subprocess.run(  # noqa: S603
    [sys.executable, "-m", "pytest", str(PRUEBAS), "-v", "--no-header"],
    cwd=RAIZ,
    capture_output=True,
    text=True,
    check=False,
)

print(pruebas.stdout[-6000:])
print(f"Código de salida: {pruebas.returncode}")

mPASSED [ 14%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_sanear_descarta_categorias_fuera_del_catalogo PASSED [ 16%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_sanear_no_elimina_filas PASSED [ 18%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_depurar_elimina_duplicados_y_filas_sin_objetivo PASSED [ 20%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_depurar_deja_el_objetivo_como_entero PASSED [ 22%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_codificar_binarias_y_ordinales PASSED [ 24%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_codificar_nominales_usa_vocabulario_fijo PASSED [ 26%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_codificar_propaga_faltantes_en_one_hot PASSED [ 28%]
tests/pipelines/feature_pipeline/test_feature_pipeline.py::test_atributos_clinicos_calculan_los_valores_esperados PASSED [ 30%]
tests/pipelines/feature_pipeline/test_feature

## 8. Conclusiones

El requerimiento queda cubierto punto por punto:

| Requisito | Dónde se implementa |
|---|---|
| Tipos esperados | esquemas Pandera de las tres etapas |
| Rangos | `RANGOS_VALIDOS` + `Check.in_range` |
| Porcentaje máximo de nulos | `validar_proporcion_nulos` (umbral 35 %) |
| Categorías válidas | `CATEGORIAS_VALIDAS` + `Check.isin` |
| Formatos de fecha | `validar_formato_fechas`, parametrizada por `COLS_FECHA` |
| Unicidad de campos clave | `validar_unicidad`, sobre el registro completo |
| Integridad entre campos | `validar_integridad_derivados`, `validar_integridad_onehot` |
| Integridad entre registros | unicidad y conteo de filas |
| Integridad entre datasets | `validar_consistencia_datasets` |
| Error claro | `ErrorDeValidacion` con etapa, columna, regla, valor y fila |
| Sin persistencia ante fallo | validaciones antes de la primera escritura |

Dos decisiones que vale la pena defender en la sustentación:

1. **El orden es la garantía, no la intención.** Que las escrituras vayan después de
   las validaciones no es un detalle de estilo: es lo que hace estructuralmente
   imposible dejar un `data/04_feature/` con un dataset a medio validar. Una versión
   que validara mientras escribe podría fallar en la última regla habiendo dejado ya
   dos archivos.

2. **Se validan los datos, no el código.** Un fallo de validación no lanza una traza
   de Python ni pide depurar el pipeline: informa de qué dato concreto está mal, para
   que la corrección ocurra en la fuente. Es lo que separa una validación útil en
   producción de un `assert` que sólo sirve al desarrollador.